# Lab 05 — Learn the Scorer

## The story

Last week you built a feed and tuned **three volume knobs by hand**: how much to care about freshness, affinity and popularity. You turned a knob, looked at a number, turned it again. That works for three knobs. It does not work for thirty, and it cannot notice that *new users and veteran users want different things*.

This week the computer turns the knobs. A tiny **neural network** (an *MLP*: multi-layer perceptron) looks at the signals for a user and a post and learns, from thousands of real likes, **how likely this user is to like this post**. Sort the posts by that number and you have a feed.

The part that makes learning possible is **backpropagation**: the algorithm that tells the network, for every number inside it, *which way to nudge it to make fewer mistakes*. You will write it yourself, in plain NumPy. No PyTorch, no autograd, no `sklearn`.

## How this notebook works

Like last week, almost everything is already implemented and explained. You **read, run and understand** it. There are **two tasks**:

1. **Task 1 — write `backward`**: the gradient of the loss with respect to every weight. A built-in *gradient check* tells you whether you got it right, and roughly where it is wrong.
2. **Task 2 — choose three numbers** (hidden neurons, learning rate, epochs) so that your trained network beats every baseline on `val`.

## The path

1. Rebuild the Week 4 signals in NumPy (vectorised) and add three new ones.
2. Turn the past into **training examples**: "at this moment the user liked this post, but not those".
3. Set up an honest exam and a set of **baselines** to beat.
4. Look at the network and its **forward pass**.
5. **Task 1:** write `backward`, and prove it with the gradient check.
6. Watch your network learn XOR, something no straight line can do.
7. Train on real likes. **Task 2:** tune three numbers.
8. See where it fails, take the final exam once, and save the model for InstaKill.

## Why movies, and not InstaKill posts?

Same reason as Week 4: to learn anything we need thousands of real likes, and InstaKill has a handful. We practise on MovieLens and then move the *trained model* (a small `.npz` file) into your `GET /feed`.

| In this notebook | In InstaKill |
|---|---|
| movie | post |
| user | user |
| rating of 4 or more | a like |
| the first time anybody rated the movie | the moment the post was created |

**Rules:** CPU only; Python standard library + NumPy; no `sklearn`, `torch`, `tensorflow`, `jax`, `autograd` or any other machine-learning library.

In [ ]:
%pip -q install numpy

In [ ]:
import csv
import json
import time
import zipfile
from collections import defaultdict, deque, namedtuple
from pathlib import Path

import numpy as np

SEED = 42
DAY = 86_400                    # seconds in a day
K = 10                          # a feed shows 10 items
CANDIDATES = 99                 # the exam: 1 liked item hidden among 99 random items

# --- Week 4 constants (unchanged) ---
HALF_LIFE_DAYS = 3 * 365        # freshness halves every 3 years
POPULARITY_DAYS = 365           # popularity counts the likes of the last year

# --- new this week ---
ACTIVITY_CAP = 500              # a user with 500+ likes counts as "fully active"
AGE_CAP_DAYS = 25 * 365         # an item older than 25 years counts as "fully old"
MIN_HISTORY = 3                 # we only learn from moments where the user already had 3+ likes
NEGATIVES_PER_LIKE = 4          # for training: every liked item is shown with 4 items the user did not like
EVAL_MOMENTS = 2_000            # at most this many exam questions per split (keeps the notebook fast)

FEATURES = ["freshness", "affinity", "popularity", "activity", "lifetime", "age"]
DATA_DIR = Path("ml-latest-small")

## Data

Same dataset as Week 4: MovieLens `latest-small`, about 100 000 ratings from 600 users. It comes from GroupLens and is **not** stored in the course repository. Read its terms of use in `README.txt` inside the zip.

**Step 1 — download it in your browser:** <https://files.grouplens.org/datasets/movielens/ml-latest-small.zip> (about 1 MB). If you still have it from Week 4, reuse it.

- **Colab:** run the next cell and pick the zip when asked.
- **Local Jupyter:** put the zip next to this notebook and run the next cell.

The cell builds a **catalogue**: all items stored in NumPy arrays, **sorted by the moment they first appeared**. That order is a trick we use later: "the items that existed at time *t*" is always the first *n* rows. A `Like` is *(user, item row, time)*.

In [ ]:
if not (DATA_DIR / "ratings.csv").exists():
    archives = sorted(Path(".").glob("ml-latest-small*.zip"))
    if not archives:
        try:
            from google.colab import files
            print("Select the ml-latest-small.zip file you downloaded.")
            files.upload()
        except ImportError:
            raise FileNotFoundError("Download ml-latest-small.zip (link above) and place it next to this notebook.")
        archives = sorted(Path(".").glob("ml-latest-small*.zip"))
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(".")
for name in ("ratings.csv", "movies.csv"):
    assert (DATA_DIR / name).exists(), f"{DATA_DIR / name} not found: upload the original ml-latest-small.zip"

with open(DATA_DIR / "ratings.csv", newline="") as f:
    EVENTS = [(int(r["userId"]), int(r["movieId"]), int(r["timestamp"]), float(r["rating"])) for r in csv.DictReader(f)]
with open(DATA_DIR / "movies.csv", newline="") as f:
    MOVIES = {int(r["movieId"]): (r["title"], tuple(g for g in r["genres"].split("|") if g != "(no genres listed)"))
              for r in csv.DictReader(f)}

first_seen = {}
for _, movie_id, ts, _ in EVENTS:
    first_seen[movie_id] = min(ts, first_seen.get(movie_id, ts))
movie_ids = sorted(first_seen, key=lambda m: (first_seen[m], m))      # oldest first
ROW = {m: row for row, m in enumerate(movie_ids)}                     # movie id -> row number

GENRES = sorted({g for m in movie_ids for g in MOVIES.get(m, ("?", ()))[1]})
genre_matrix = np.zeros((len(movie_ids), len(GENRES)))                # row i has 1.0 in the columns of item i's genres
for row, m in enumerate(movie_ids):
    for g in MOVIES.get(m, ("?", ()))[1]:
        genre_matrix[row, GENRES.index(g)] = 1.0

Catalogue = namedtuple("Catalogue", "titles first_seen genres n_genres")
CATALOGUE = Catalogue(
    titles=[MOVIES.get(m, ("?", ()))[0] for m in movie_ids],
    first_seen=np.array([first_seen[m] for m in movie_ids]),
    genres=genre_matrix,
    n_genres=np.maximum(genre_matrix.sum(axis=1), 1.0),               # never divide by 0
)

Like = namedtuple("Like", "user_id row ts")
LIKES = sorted((Like(u, ROW[m], ts) for u, m, ts, rating in EVENTS if rating >= 4.0),
               key=lambda like: (like.ts, like.user_id, like.row))
N_USERS = max(like.user_id for like in LIKES) + 1

# Same time cuts as Week 4: 70% of likes are "train", the next 15% "val", the last 15% "test"
timestamps = np.array([like.ts for like in LIKES])
T_VAL, T_TEST = int(np.quantile(timestamps, 0.70)), int(np.quantile(timestamps, 0.85))
print(f"{len(LIKES):,} likes, {len(movie_ids):,} items, {len({l.user_id for l in LIKES})} users, {len(GENRES)} genres")
print(f"catalogue arrays: first_seen {CATALOGUE.first_seen.shape}, genres {CATALOGUE.genres.shape}")

## Step 1 — Signals, again, but in NumPy

Last week each signal was a small Python function that you called once per item. With thousands of items per user that is a lot of Python loops. NumPy can compute the same number for **all items at once**; this is called **vectorisation**, and it is the whole point of NumPy.

You already know three signals from Week 4. We add three more that a hand-tuned formula cannot easily use:

| Signal | Plain question | Comes from |
|---|---|---|
| **freshness** | how new is the item? | Week 4: `0.5 ** (age / half-life)` |
| **affinity** | does it match the genres this user already likes? | Week 4: genre overlap |
| **popularity** | how many people liked it in the last year? | Week 4: recent likes ÷ most-liked item |
| **activity** *(new)* | how much do we know about this user? | `log(1 + user's likes)`, scaled to 0–1 |
| **lifetime** *(new)* | how many likes has the item collected since forever? | `log(1 + all likes)`, scaled to 0–1 |
| **age** *(new)* | how old is the item, on a plain linear scale? | age ÷ 25 years, capped at 1 |

Why add these? *activity* lets the network treat new users (we know nothing about them) differently from veterans: a **cold start** signal. *lifetime* separates classics from flashes in the pan, which "popularity of the last year" cannot. *age* next to *freshness* gives the network two views of the same fact, from which it can learn a more complicated shape (for example "not brand new, not ancient"). A single weight per signal can't express any of that; a hidden layer can.

**Same two rules as Week 4.** *Only the past*: the signals at time `now` use only likes before `now`. *Same scale*: every signal lands between 0 and 1.

`LiveSignals` keeps all the counting in NumPy arrays and is built to be fed likes **in time order** with `add_like`. It is the vectorised, incremental version of Week 4's `build_signals`. Read it; the comments explain each line.

In [ ]:
class LiveSignals:
    """What we know about the world right now. Feed it likes in time order with add_like()."""

    def __init__(self, catalogue, n_users):
        self.catalogue = catalogue
        self.genre_counts = np.zeros((n_users, len(GENRES)))   # row u: how many liked items of each genre user u has
        self.n_likes = np.zeros(n_users)                       # likes per user
        self.liked = defaultdict(set)                          # user -> set of item rows already liked
        self.recent = np.zeros(len(catalogue.titles))          # per item: likes in the last POPULARITY_DAYS days
        self.lifetime = np.zeros(len(catalogue.titles))        # per item: likes ever
        self._window = deque()                                 # (ts, row) of the likes counted in `recent`

    def add_like(self, like):
        self.genre_counts[like.user_id] += self.catalogue.genres[like.row]   # one vector addition, not a loop over genres
        self.n_likes[like.user_id] += 1
        self.liked[like.user_id].add(like.row)
        self.recent[like.row] += 1
        self.lifetime[like.row] += 1
        self._window.append((like.ts, like.row))

    def advance(self, now):
        """Forget the likes that are older than POPULARITY_DAYS days."""
        while self._window and self._window[0][0] < now - POPULARITY_DAYS * DAY:
            _, row = self._window.popleft()
            self.recent[row] -= 1

    def features(self, user_id, rows, now):
        """The 6 signals of user `user_id` for the items `rows` (an int array), as a (len(rows), 6) array."""
        cat = self.catalogue
        age_days = (now - cat.first_seen[rows]) / DAY
        freshness = 0.5 ** (age_days / HALF_LIFE_DAYS)

        counts = self.genre_counts[user_id]
        total = counts.sum()
        if total > 0:
            affinity = (cat.genres[rows] @ counts) / (total * cat.n_genres[rows])   # a matrix-vector product
        else:
            affinity = np.zeros(len(rows))                                           # nothing known about this user

        popularity = self.recent[rows] / max(self.recent.max(), 1)
        activity = np.full(len(rows), min(1.0, np.log1p(self.n_likes[user_id]) / np.log1p(ACTIVITY_CAP)))
        lifetime = np.log1p(self.lifetime[rows]) / np.log1p(max(self.lifetime.max(), 1))
        age = np.minimum(1.0, age_days / AGE_CAP_DAYS)
        return np.column_stack([freshness, affinity, popularity, activity, lifetime, age])

## Self-check 1 — do the signals work?

A tiny hand-made world where we know the right answer. It reuses the Week 4 example: user 7 liked three dramas and one comedy, so a new drama gets affinity `3 / (4 × 1) = 0.75`. It also checks that likes older than a year leave `popularity` but stay in `lifetime`.

In [ ]:
def tiny_catalogue():
    # 4 liked items (3 Drama, 1 Comedy), then a Drama (row 4), a Drama+Comedy (row 5), and a Drama that is not popular (row 6)
    genre_sets = [("Drama",), ("Drama",), ("Drama",), ("Comedy",), ("Drama",), ("Drama", "Comedy"), ("Drama",)]
    g = np.zeros((len(genre_sets), len(GENRES)))
    for row, gs in enumerate(genre_sets):
        for name in gs:
            g[row, GENRES.index(name)] = 1.0
    return Catalogue(titles=[f"item {r}" for r in range(len(genre_sets))], first_seen=np.zeros(len(genre_sets), dtype=np.int64),
                     genres=g, n_genres=np.maximum(g.sum(axis=1), 1.0))

NOW = HALF_LIFE_DAYS * DAY                      # exactly one half-life after the items appeared
tiny = LiveSignals(tiny_catalogue(), n_users=10)
tiny.add_like(Like(8, 6, NOW - 400 * DAY))                   # row 6 was liked once, more than a year ago (likes must be added in time order)
for row in range(4):
    tiny.add_like(Like(7, row, NOW - 10 * DAY))              # user 7: 3 x Drama, 1 x Comedy
tiny.add_like(Like(8, 4, NOW - 10 * DAY))                    # two other users liked row 4
tiny.add_like(Like(9, 4, NOW - 5 * DAY))
tiny.advance(NOW)

f = tiny.features(7, np.array([4, 5, 0]), NOW)
assert abs(f[0, 0] - 0.5) < 1e-9, "freshness after exactly one half-life must be 0.5"
assert abs(f[0, 1] - 0.75) < 1e-9 and abs(f[1, 1] - 0.5) < 1e-9, "affinity: 3/(4*1) for a Drama, (3+1)/(4*2) for Drama+Comedy"
assert abs(f[0, 2] - 1.0) < 1e-9 and abs(f[2, 2] - 0.5) < 1e-9, "popularity: row 4 has the most recent likes (2), row 0 has 1"
assert abs(f[0, 3] - np.log1p(4) / np.log1p(ACTIVITY_CAP)) < 1e-9, "activity of a user with 4 likes"
assert abs(f[0, 4] - 1.0) < 1e-9, "lifetime of the most-liked item is 1.0"
assert abs(f[0, 5] - HALF_LIFE_DAYS / (25 * 365)) < 1e-9, "age is linear: 3 years / 25 years"
assert tiny.features(3, np.array([4]), NOW)[0, 1] == 0.0, "a user with no likes has affinity 0"
g = tiny.features(7, np.array([6]), NOW)[0]
assert g[2] == 0.0 and g[4] > 0.0, "the old like left `popularity` (expired) but stays in `lifetime`"
print("Self-check 1 passed.")

## Why NumPy? The same answer, 100× faster

The cell below computes **affinity for all ~9 700 items** for one user in two ways: with the plain Python loop style of Week 4, and with the vectorised line inside `LiveSignals.features`. It first checks that both give the same numbers, then times them.

Read the two timings. In InstaKill the feed is computed on every request, so this gap is the difference between "instant" and "the page hangs".

In [ ]:
signals_demo = LiveSignals(CATALOGUE, N_USERS)
for like in LIKES[:20_000]:
    signals_demo.add_like(like)
demo_user = max(range(N_USERS), key=lambda u: signals_demo.n_likes[u])
demo_now = LIKES[20_000].ts
all_rows = np.arange(int(np.searchsorted(CATALOGUE.first_seen, demo_now)))      # items that existed at demo_now


def affinity_loop(user_id, rows):
    """Week 4 style: one Python call per item."""
    counts = signals_demo.genre_counts[user_id]
    total = counts.sum()
    out = []
    for row in rows:
        genres = [j for j in range(len(GENRES)) if CATALOGUE.genres[row, j] == 1.0]
        matching = sum(counts[j] for j in genres)
        out.append(matching / (total * max(len(genres), 1)))
    return np.array(out)


def affinity_vectorised(user_id, rows):
    counts = signals_demo.genre_counts[user_id]
    return (CATALOGUE.genres[rows] @ counts) / (counts.sum() * CATALOGUE.n_genres[rows])


assert np.allclose(affinity_loop(demo_user, all_rows), affinity_vectorised(demo_user, all_rows)), "the two versions must agree"

def mean_ms(fn, repeats=5):
    started = time.perf_counter()
    for _ in range(repeats):
        fn()
    return 1000 * (time.perf_counter() - started) / repeats

loop_ms = mean_ms(lambda: affinity_loop(demo_user, all_rows), repeats=2)
vec_ms = mean_ms(lambda: affinity_vectorised(demo_user, all_rows), repeats=50)
print(f"affinity of {len(all_rows):,} items:  python loop {loop_ms:8.1f} ms   |   vectorised {vec_ms:6.2f} ms   |   {loop_ms / vec_ms:,.0f}x faster")

## Step 2 — Training examples: "at this moment the user liked this one, not those"

A network learns from examples with an answer. Our examples are built from **moments**: every like in the data is a moment when a user opened the app and picked something. For that moment we build:

- one **positive** row: the signals of the item the user liked (label `1`),
- several **negative** rows: signals of items that existed at that moment and that the user did **not** like (label `0`), picked at random.

The signals are computed from **what was known just before that like**, exactly the information the real app would have had. That is why `LiveSignals` is fed one like at a time: first we build the rows for the moment, then we add the like to the history. A like never sees its own future.

Two details worth a sentence each:

- Items the user liked *at any time* are never used as negatives, so we do not tell the network "this is bad" about something the user ends up loving.
- Only moments where the user already had `MIN_HISTORY` likes are used, and the liked item must have existed before the moment.

**Splits by time**, as in Week 4: moments before `T_VAL` train the network, those between `T_VAL` and `T_TEST` are `val` (use freely while tuning), and later ones are `test` (look **once** at the end).

**The exam.** For `val` and `test` we hide the liked item among **99 random items** that existed at that moment and ask the model to score all 100. A good model puts the real like near the top. We report **NDCG@10** and **HitRate@10**, as last week, so higher is better and `1.0` is perfect. A model that guesses scores about `0.04`. Because every moment is one exam question, we get ~1 800 questions per split instead of the handful of users of Week 4, so the numbers are much steadier.

*These numbers are not comparable to Week 4's.* The exam changed; so to compare fairly, we re-score the Week 4 formulas on the new exam below.

In [ ]:
class Dataset:
    pass


def build_examples(likes, catalogue, n_users, seed=SEED):
    rng = np.random.default_rng(seed)
    ever_liked = defaultdict(set)
    for like in likes:
        ever_liked[like.user_id].add(like.row)

    def pick(low, high):                       # a fixed random subset of the moments in [low, high)
        idx = [k for k, like in enumerate(likes) if low <= like.ts < high]
        return set(rng.choice(idx, min(EVAL_MOMENTS, len(idx)), replace=False).tolist())

    exam = {"val": pick(T_VAL, T_TEST), "test": pick(T_TEST, 1 << 62)}

    signals = LiveSignals(catalogue, n_users)
    data = Dataset()
    train_x, train_y = [], []
    data.train_meta = []
    for name in exam:
        setattr(data, f"{name}_x", []); setattr(data, f"{name}_rows", []); setattr(data, f"{name}_meta", [])

    for k, like in enumerate(likes):
        signals.advance(like.ts)
        part = "train" if like.ts < T_VAL else ("val" if like.ts < T_TEST else "test")
        n_exist = int(np.searchsorted(catalogue.first_seen, like.ts, side="left"))   # items that existed before this moment
        wanted = NEGATIVES_PER_LIKE if part == "train" else (CANDIDATES if k in exam[part] else 0)
        if wanted and signals.n_likes[like.user_id] >= MIN_HISTORY and catalogue.first_seen[like.row] < like.ts and n_exist >= 200:
            negatives = []
            while len(negatives) < wanted:
                row = int(rng.integers(0, n_exist))
                if row not in ever_liked[like.user_id] and row not in negatives:
                    negatives.append(row)
            rows = np.array([like.row] + negatives)                  # the liked item is always entry 0
            x = signals.features(like.user_id, rows, like.ts)
            if part == "train":
                train_x.append(x); train_y.append(np.r_[1.0, np.zeros(wanted)]); data.train_meta.append(like)
            else:
                getattr(data, f"{part}_x").append(x); getattr(data, f"{part}_rows").append(rows); getattr(data, f"{part}_meta").append(like)
        signals.add_like(like)                                       # only now does the like become history

    data.train_x, data.train_y = np.vstack(train_x), np.concatenate(train_y)
    for name in exam:
        setattr(data, f"{name}_x", np.stack(getattr(data, f"{name}_x")))        # shape (moments, 100, 6)
        setattr(data, f"{name}_rows", np.stack(getattr(data, f"{name}_rows")))  # shape (moments, 100)
    return data


started = time.perf_counter()
DATA = build_examples(LIKES, CATALOGUE, N_USERS)
print(f"built in {time.perf_counter() - started:.1f} s")
print(f"train: {DATA.train_x.shape[0]:,} rows ({DATA.train_y.mean():.0%} positive), shape {DATA.train_x.shape}")
print(f"val:   {DATA.val_x.shape[0]:,} exam questions, shape {DATA.val_x.shape}  (moments, 1 liked + 99 random, signals)")
print(f"test:  {DATA.test_x.shape[0]:,} exam questions, shape {DATA.test_x.shape}")

## Self-check 2 — is it honest? (no peeking at the future)

Two checks.

1. **Same numbers from scratch.** Pick a moment, build a *new* `LiveSignals` fed only with the likes that came before it, and compute the signals of the liked item. They must equal the numbers stored in the training set. This proves the incremental bookkeeping (counts, the sliding one-year window) matches a from-scratch computation.
2. **Time order.** Training moments come strictly before `val`, and `val` strictly before `test`.

In [ ]:
for position in (3, 700, 12_000, 29_000):
    like = DATA.train_meta[position]
    index = LIKES.index(like)
    fresh = LiveSignals(CATALOGUE, N_USERS)
    for earlier in LIKES[:index]:                       # only the past
        fresh.add_like(earlier)
    fresh.advance(like.ts)
    stored = DATA.train_x[position * (1 + NEGATIVES_PER_LIKE)]          # first row of the moment = the liked item
    recomputed = fresh.features(like.user_id, np.array([like.row]), like.ts)[0]
    assert np.allclose(stored, recomputed), f"moment {position}: stored {stored} vs recomputed {recomputed}"

assert max(l.ts for l in DATA.train_meta) < T_VAL <= min(l.ts for l in DATA.val_meta) and max(l.ts for l in DATA.val_meta) < T_TEST <= min(l.ts for l in DATA.test_meta)
assert np.isfinite(DATA.train_x).all() and DATA.train_x.min() >= 0.0 and DATA.train_x.max() <= 1.0, "signals must stay between 0 and 1"
print("Self-check 2 passed.")

## Step 3 — The exam and the baselines

**Ranks.** For each exam question we have 100 scores, the liked item in column 0. Its **rank** is how many of the 99 random items scored *at least as high* (0 = best; ties count against us, so a model that gives everything the same score gets no credit). NDCG@10 and HitRate@10 follow from the rank:

- **HitRate@10** — the share of questions where the liked item landed in the top 10;
- **NDCG@10** — same, but a rank-0 hit is worth more than a rank-9 hit.

Notice there are no loops over questions: `scores[:, 1:] >= scores[:, :1]` compares **every question at once** by *broadcasting*, a `(moments, 99)` array against a `(moments, 1)` column.

In [ ]:
def positive_ranks(scores):
    """scores: (moments, 100), the liked item in column 0. Returns, per moment, how many random items scored >= it."""
    return (scores[:, 1:] >= scores[:, :1]).sum(axis=1)


def ndcg_at_k(ranks, k=K):
    return np.where(ranks < k, 1.0 / np.log2(ranks + 2), 0.0)       # one value per moment


def hit_at_k(ranks, k=K):
    return (ranks < k).astype(float)


def exam(score_fn, split):
    """score_fn: raw signals (n, 6) -> one score per row (higher = better). split: 'val' or 'test'."""
    x = getattr(DATA, f"{split}_x")
    moments, candidates, n_features = x.shape
    scores = score_fn(x.reshape(-1, n_features)).reshape(moments, candidates)
    ranks = positive_ranks(scores)
    return {"ndcg": float(ndcg_at_k(ranks).mean()), "hit_rate": float(hit_at_k(ranks).mean()),
            "ndcg_per_moment": ndcg_at_k(ranks), "scores": scores, "ranks": ranks}


# Self-check 3 — the metrics, on three hand-made questions
toy = np.array([[0.9, 0.1, 0.2, 0.3],      # liked item scores highest              -> rank 0
                [0.1, 0.9, 0.8, 0.7],      # three items beat it                    -> rank 3
                [0.5, 0.5, 0.5, 0.1]])     # two items TIE with it (ties count)     -> rank 2
assert positive_ranks(toy).tolist() == [0, 3, 2]
assert np.allclose(ndcg_at_k(positive_ranks(toy)), [1.0, 1 / np.log2(5), 0.5])
assert hit_at_k(positive_ranks(toy), k=3).tolist() == [1.0, 0.0, 1.0]
random_scores = np.random.default_rng(0).random((5_000, 100))
assert abs(ndcg_at_k(positive_ranks(random_scores)).mean() - 0.045) < 0.01, "random scores should get NDCG@10 of about 0.05"
print("Self-check 3 passed.  (Guessing gets NDCG@10 of about 0.05: that is the floor.)")

**Baselines.** A model is only worth its cost if it beats something simple. We give you four, all scored on the same exam:

| Baseline | Score of an item |
|---|---|
| **chronological** | its freshness (newest first) |
| **popularity** | its popularity signal (the same ordering for everybody) |
| **Week 4 formula** | `w_fresh × freshness + w_aff × affinity + w_pop × popularity` with *default* weights `(1, 1, 1)` |
| **your Week 4 weights** | the same formula with the weights *you* chose last week: paste them below |
| **linear (least squares)** | the best straight-line scorer over **all six** signals, fitted to the training rows by one call to `np.linalg.lstsq` |

The last one matters most. It is a "model" too (six learned numbers), but with no hidden layer. If your network cannot beat a straight line, its hidden layer is dead weight, and saying so is a valid result.

Signals live on different scales (most `popularity` values are tiny, `activity` is large), which makes learning slow and fussy. The standard remedy is **standardisation**: subtract the training mean and divide by the training standard deviation, per signal. We compute `MU` and `SD` once on the training rows and reuse them everywhere. **They are part of the model**: InstaKill must apply the very same numbers.

In [ ]:
MU, SD = DATA.train_x.mean(axis=0), DATA.train_x.std(axis=0) + 1e-9      # one number per signal (axis 0 = across rows)
standardize = lambda features: (features - MU) / SD                      # broadcasting: (n, 6) - (6,)

# ======== paste the weights you chose in Week 4 (leave as is if you do not have them) ========
WEEK4_WEIGHTS = {"freshness": 1.0, "affinity": 1.0, "popularity": 1.0}
# ==============================================================================================


def weighted_scorer(weights):
    vector = np.zeros(len(FEATURES))
    vector[:3] = [weights["freshness"], weights["affinity"], weights["popularity"]]
    return lambda features: features @ vector


design = np.c_[standardize(DATA.train_x), np.ones(len(DATA.train_x))]            # signals + a constant column for the intercept
linear_weights = np.linalg.lstsq(design, DATA.train_y, rcond=None)[0]
linear_scorer = lambda features: np.c_[standardize(features), np.ones(len(features))] @ linear_weights

BASELINES = {
    "chronological": lambda features: features[:, 0],
    "popularity": lambda features: features[:, 2],
    "Week 4 default (1, 1, 1)": weighted_scorer({"freshness": 1.0, "affinity": 1.0, "popularity": 1.0}),
    "your Week 4 weights": weighted_scorer(WEEK4_WEIGHTS),
    "linear (least squares)": linear_scorer,
}
baseline_val = {name: exam(fn, "val")["ndcg"] for name, fn in BASELINES.items()}
for name, value in baseline_val.items():
    print(f"{name:26s} val NDCG@10 = {value:.4f}  {'█' * round(80 * value)}")

Read the table. Two things to notice, and to remember when you write the failure cases:

- The Week 4 formula with default weights may be *worse* than plain popularity here: a signal that helped last week can hurt under a different exam.
- The straight line over six signals is clearly better than anything hand-made. That is the bar for the network.

## Step 4 — The network

```text
 6 signals ──▶ [ W1, b1 ] ──▶ H hidden neurons ──▶ [ W2, b2 ] ──▶ one raw score z ──▶ sigmoid ──▶ P(like)
   (N, 6)       (6,H) (H,)     tanh squashes         (H,1) (1,)        (N,)                  between 0 and 1
```

A **neuron** is just "a weighted sum of its inputs, then a squashing function". The **hidden layer** is `H` such neurons side by side; because each one can pick a different mix of the signals and the squashing is *non-linear*, together they can draw curves a single weighted sum cannot. The output neuron then mixes the hidden ones into one score. With `H` hidden neurons the network has `6·H + H + H + 1` numbers to learn.

The **forward pass** (given) is two matrix products:

| Step | Formula | Shape |
|---|---|---|
| hidden pre-activation | `Z1 = X @ W1 + b1` | `(N, 6) @ (6, H) + (H,)` → `(N, H)` |
| hidden activation | `A1 = tanh(Z1)` | `(N, H)` |
| raw score ("logit") | `z = A1 @ W2 + b2` | `(N, H) @ (H, 1) + (1,)` → `(N, 1)` → `(N,)` |
| probability | `p = sigmoid(z) = 1 / (1 + e^-z)` | `(N,)` |

**The loss** says *how wrong* the network is, as one number: the average **binary cross-entropy** `−[ y·log p + (1−y)·log(1−p) ]`. It is small when `p` is close to the true label and grows without bound for a confident wrong answer. We compute it directly from `z` in a numerically safe form (`log(1+e^z) − y·z`); a network that knows nothing (`p = 0.5` everywhere) has loss `ln 2 ≈ 0.693`.

**Learning** means: compute the loss, compute the **gradient** (for every number in `W1, b1, W2, b2`, how much the loss would change if that number grew a little), and move every number a small step *against* its gradient. Repeat on small random batches. The gradient is the one piece missing. That is your task.

In [ ]:
def init_params(n_features, hidden, seed=SEED):
    """Small random weights (never all zeros: identical neurons would learn identical things) and zero biases."""
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0.0, 1 / np.sqrt(n_features), (n_features, hidden)),
            "b1": np.zeros(hidden),
            "W2": rng.normal(0.0, 1 / np.sqrt(hidden), (hidden, 1)),
            "b2": np.zeros(1)}


def forward(params, X):
    """X: (N, n_features). Returns the raw scores z (N,) and a cache with what backward() will need."""
    Z1 = X @ params["W1"] + params["b1"]
    A1 = np.tanh(Z1)
    z = (A1 @ params["W2"] + params["b2"])[:, 0]
    return z, {"X": X, "A1": A1, "z": z}


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def bce_loss(z, y):
    """Mean binary cross-entropy, computed from the raw score z (stable for large |z|)."""
    return float(np.mean(np.logaddexp(0.0, z) - y * z))


def count_parameters(params):
    return int(sum(value.size for value in params.values()))


# Self-check 4 — shapes, and the loss of a network that knows nothing
rng_check = np.random.default_rng(0)
x_check, y_check = rng_check.normal(size=(8, 6)), (rng_check.random(8) < 0.5).astype(float)
params_check = init_params(6, 5)
z_check, cache_check = forward(params_check, x_check)
assert z_check.shape == (8,) and cache_check["A1"].shape == (8, 5)
assert count_parameters(params_check) == 6 * 5 + 5 + 5 + 1
zero_params = {name: np.zeros_like(value) for name, value in params_check.items()}
assert abs(bce_loss(forward(zero_params, x_check)[0], y_check) - np.log(2)) < 1e-12, "a network that knows nothing has loss ln 2"
print("Self-check 4 passed.  Parameters for 6 signals and 5 hidden neurons:", count_parameters(params_check))

## Task 1 — Write `backward`

`backward(params, cache, y)` must return the gradient of the **mean loss** with respect to every parameter: a dict with the same keys **and the same shapes** as `params` (`W1`, `b1`, `W2`, `b2`). `cache` holds what `forward` computed (`X`, `A1`, `z`); `y` holds the `N` labels.

**The idea in one paragraph.** The loss is built from a chain of simple steps: `X → Z1 → A1 → z → loss`. The **chain rule** says that the effect of an early value on the loss is the product of the effects of each step along the way. So we walk the chain **backwards** (that is the "back" in *back*propagation): start with how the loss changes with `z`, then ask how `z` changes with `A1` and `W2`, then how `A1` changes with `Z1`, and so on. Each step reuses the result of the one after it, so the whole gradient costs about as much as one forward pass.

**Two things always help.**

- The gradient of an array has **the same shape as the array**. If you compute something of shape `(H, N)` where `(N, H)` is needed, you have a transpose to add or a product in the wrong order.
- The loss is a **mean over `N` rows**, so each row contributes a factor `1/N`. A bias is shared by all rows, so its gradient **sums** over the rows.

<details>
<summary><b>Hint 1</b> — a plan (open if you do not know where to start)</summary>

Write the forward pass as a list: `Z1`, `A1`, `z`, `p = sigmoid(z)`, `loss`. Then work from the end of that list to the start. For each value ask: "how much does the loss change if this value changes a little?", and call that `dValue`. You will need `dz`, then `dW2` and `db2` and `dA1`, then `dZ1`, then `dW1` and `db1`.
</details>

<details>
<summary><b>Hint 2</b> — the pieces (open only if you are still stuck after 15 minutes)</summary>

- For the mean binary cross-entropy, the gradient with respect to each raw score is `dz = (p − y) / N`, with `p = sigmoid(z)`. One value per row.
- For a product `C = A @ B` with upstream gradient `dC`: `dA = dC @ B.T` and `dB = A.T @ dC`.
- For an addition of a bias, sum the upstream gradient over the rows (`axis=0`).
- For `A = tanh(Z)`: `dZ = dA * (1 − A²)`.
</details>

<details>
<summary><b>Hint 3</b> — debugging with the gradient check</summary>

The check below compares your gradient with a slow but trustworthy numerical estimate, **one parameter group at a time**. If `W2` and `b2` pass but `W1` and `b1` fail, the output layer is right and the mistake is in the hidden layer (the `tanh` step or a transpose). If your gradient is off by exactly a factor `N`, the `1/N` is missing (or applied twice); the check message says which.
</details>

In [ ]:
def backward(params, cache, y):
    """Gradient of the mean binary cross-entropy w.r.t. every parameter.

    params: dict W1 (n_features, H), b1 (H,), W2 (H, 1), b2 (1,)
    cache:  dict from forward(): X (N, n_features), A1 (N, H), z (N,)
    y:      (N,) labels, 0.0 or 1.0
    returns: dict with the same keys and shapes as params
    """
    X, A1, z = cache["X"], cache["A1"], cache["z"]
    n = y.shape[0]
    p = sigmoid(z)
    dz = (p - y) / n
    dW2 = A1.T @ dz[:, None]
    db2 = np.array([dz.sum()])
    dA1 = dz[:, None] @ params["W2"].T
    dZ1 = dA1 * (1.0 - A1 * A1)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

## Self-check 5 — the gradient check

How do we know a gradient is right without a reference? We estimate it **numerically**: nudge one parameter up by a tiny `ε`, nudge it down by `ε`, measure how the loss changed, and divide by `2ε`. This is slow (two forward passes *per parameter*), so we use it only as a test, on a small network with 25 numbers; training uses your fast `backward`.

The check reports, for each parameter group, the largest **relative error** between your gradient and the numerical one. A correct `backward` scores around `1e-9`; the bar is `1e-6`. The cell tells you which group is wrong and, for the most common mistakes, what the likely cause is.

In [ ]:
def numerical_gradient(params, X, y, name, eps=1e-6):
    grad = np.zeros_like(params[name])
    for index in np.ndindex(*params[name].shape):
        original = params[name][index]
        params[name][index] = original + eps
        loss_plus = bce_loss(forward(params, X)[0], y)
        params[name][index] = original - eps
        loss_minus = bce_loss(forward(params, X)[0], y)
        params[name][index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def check_gradients(backward_fn, hidden=3, batch=16, tolerance=1e-6):
    rng = np.random.default_rng(1)
    params = init_params(len(FEATURES), hidden, seed=1)
    params["b1"], params["b2"] = rng.normal(size=hidden), rng.normal(size=1)       # non-zero biases, so a wrong bias gradient shows up
    X, y = rng.normal(size=(batch, len(FEATURES))), (rng.random(batch) < 0.4).astype(float)
    grads = backward_fn(params, forward(params, X)[1], y)

    assert isinstance(grads, dict) and set(grads) == set(params), "backward must return a dict with the keys W1, b1, W2, b2"
    report, failed = [], []
    for name in params:
        assert grads[name].shape == params[name].shape, f"{name}: gradient has shape {grads[name].shape}, expected {params[name].shape}"
        numeric = numerical_gradient(params, X, y, name)
        error = np.max(np.abs(grads[name] - numeric) / np.maximum(1e-8, np.abs(grads[name]) + np.abs(numeric)))
        report.append((name, error))
        if error > tolerance:
            failed.append(name)
            if np.allclose(grads[name] / batch, numeric, atol=1e-6):
                print(f"  {name}: your gradient is {batch}x too big: did you forget to divide by N = {batch}? (the loss is a mean over N rows)")
            elif np.allclose(grads[name] * batch, numeric, atol=1e-6):
                print(f"  {name}: your gradient is {batch}x too small: you divided by N more than once.")
            elif not np.any(grads[name]):
                print(f"  {name}: your gradient is all zeros: it is not implemented yet.")
    for name, error in report:
        print(f"{name}: max relative error {error:9.2e}   {'ok' if name not in failed else 'WRONG'}")
    assert not failed, ("gradient check failed for " + ", ".join(failed) +
                        (". W2 and b2 are fine: look at the hidden layer (the tanh step, a transpose)." if set(failed) <= {"W1", "b1"} else
                         ". Start with W2/b2 (the output layer), then continue backwards."))
    return True


GRADIENTS_OK = check_gradients(backward)
print("Self-check 5 passed: your backward is correct.")

## Sanity check — the problem no straight line can solve (XOR)

Before the real data, the smallest problem that *needs* a hidden layer. Four points; the label is 1 when **exactly one** of the two inputs is 1:

| x1 | x2 | label |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No weighted sum `w1·x1 + w2·x2 + b` can separate these (try it on paper). The best straight line is stuck at predicting `0.5` everywhere. A network with a hidden layer can. `train` below is the training loop used for the real task: **mini-batch gradient descent**. Each step takes a small random batch of rows, computes the gradient with *your* `backward`, and moves every parameter by `−learning_rate × gradient`. One pass over all rows is an **epoch**.

In [ ]:
def train(params, X, y, epochs, learning_rate, batch_size=128, seed=SEED, monitor=None):
    """Mini-batch gradient descent. Returns the trained parameters (a copy) and one record per epoch."""
    params = {name: value.copy() for name, value in params.items()}
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(X))                           # a new shuffle every epoch
        for start in range(0, len(X), batch_size):
            batch = order[start:start + batch_size]
            _, cache = forward(params, X[batch])
            grads = backward(params, cache, y[batch])
            for name in params:
                params[name] -= learning_rate * grads[name]       # a small step against the gradient
        record = {"epoch": epoch, "loss": bce_loss(forward(params, X)[0], y)}
        if monitor is not None:
            record["val_ndcg"] = monitor(params)
        history.append(record)
    return params, history


assert GRADIENTS_OK, "finish Task 1 first: the gradient check above must pass"
xor_x = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
xor_y = np.array([0.0, 1.0, 1.0, 0.0])
line = np.linalg.lstsq(np.c_[xor_x, np.ones(4)], xor_y, rcond=None)[0]
print("best straight line predicts:", np.round(np.c_[xor_x, np.ones(4)] @ line, 2), "  (always 0.5: it cannot see the pattern)")

xor_params, xor_history = train(init_params(2, 4, seed=3), xor_x, xor_y, epochs=3000, learning_rate=0.5, batch_size=4)
xor_p = sigmoid(forward(xor_params, xor_x)[0])
print("network predicts:          ", np.round(xor_p, 2), f"  (loss {xor_history[-1]['loss']:.4f})")
assert np.all((xor_p > 0.5) == (xor_y == 1.0)), "the network should get all four XOR points right: check backward"
print("The hidden layer solved XOR.")

## Step 5 — Train on the real likes

Same loop, real rows: ~150 000 training rows of 6 standardised signals. Each epoch the cell also scores the network on the `val` exam, so you can watch **validation NDCG@10** (the number we care about) next to the **training loss** (the number the network minimises). They usually move together; when training loss keeps falling and validation stops improving, the network has started to memorise. That is **overfitting**.

## Task 2 — Choose three numbers

| Knob | What it does | Too small | Too big |
|---|---|---|---|
| `HIDDEN` | how many neurons the hidden layer has (its capacity) | cannot learn curves | slower; may memorise the training rows |
| `LEARNING_RATE` | how big a step each batch takes | learning crawls | the loss jumps around or explodes |
| `EPOCHS` | how many passes over the training rows | stops before it has learned | wastes time; may overfit |

The starting values below are deliberately poor. Run the cell, read the printed learning curve, and change **one number at a time**. A handful of runs is enough (each takes a few seconds). Tune on `val` only; `test` is for the end.

**Goal:** a network whose `val` NDCG@10 is higher than **every baseline** in the table above, including the straight line.

In [ ]:
def network_scorer(params):
    """Rank by the raw score z: sigmoid is increasing, so it never changes an order."""
    return lambda features: forward(params, standardize(features))[0]


def val_ndcg_of(params):
    return exam(network_scorer(params), "val")["ndcg"]


def show_history(history, every=1):
    for record in history:
        if record["epoch"] % every == 0 or record["epoch"] == len(history):
            print(f"epoch {record['epoch']:3d}  train loss {record['loss']:.4f}  val NDCG@10 {record['val_ndcg']:.4f}  {'█' * round(80 * record['val_ndcg'])}")


# ============================= TODO: change these three numbers =============================
HIDDEN = 8               # hidden neurons
LEARNING_RATE = 0.01     # step size
EPOCHS = 30              # passes over the training rows
# ============================================================================================

assert GRADIENTS_OK, "finish Task 1 first"
started = time.perf_counter()
model, history = train(init_params(len(FEATURES), HIDDEN), standardize(DATA.train_x), DATA.train_y,
                       epochs=EPOCHS, learning_rate=LEARNING_RATE, monitor=val_ndcg_of)
print(f"trained {count_parameters(model)} parameters in {time.perf_counter() - started:.1f} s\n")
show_history(history, every=max(1, EPOCHS // 15))
print(f"\nbaselines on val:  best simple baseline = {max(baseline_val.values()):.4f}  (linear {baseline_val['linear (least squares)']:.4f})")

In [ ]:
mine_val = exam(network_scorer(model), "val")["ndcg"]
bar = max(baseline_val.values())
assert mine_val > baseline_val["popularity"], f"not even better than plain popularity yet ({mine_val:.4f} vs {baseline_val['popularity']:.4f}): the network has not learned enough. Read the learning curve: is the loss still falling?"
assert mine_val > bar, f"better than popularity, but not yet better than every baseline ({mine_val:.4f} vs {bar:.4f}): change one knob and re-run the cell above"
print(f"Checkpoint passed: val NDCG@10 = {mine_val:.4f}  (best baseline {bar:.4f}, linear {baseline_val['linear (least squares)']:.4f}).")

## Where does it fail?

An average hides who is served badly. Two views:

1. **By how much we know about the user.** Moments are split in three equal groups by the `activity` signal. Who does the network serve best, and who worst? Compare with popularity in each group. (Every InstaKill user is a newcomer on day one: the *cold-start* problem.)
2. **The three worst moments.** For each: which item the user really liked, how many random items the model preferred to it, what the model's favourite was, and the signals involved.

Read the output, then write **one plain sentence per case** in `RESULTS`. A good sentence says *why* (for example "a very old film with almost no recent likes: freshness and popularity both say 'boring'"), not just *that* it failed.

In [ ]:
final_val = exam(network_scorer(model), "val")
pop_val = exam(BASELINES["popularity"], "val")
activity = DATA.val_x[:, 0, FEATURES.index("activity")]
bucket = np.digitize(activity, np.quantile(activity, [1 / 3, 2 / 3]))
print(f"{'users':24s} {'moments':>8s} {'network':>9s} {'popularity':>11s}")
for b, name in enumerate(["least active third", "middle third", "most active third"]):
    m = bucket == b
    print(f"{name:24s} {m.sum():8d} {final_val['ndcg_per_moment'][m].mean():9.3f} {pop_val['ndcg_per_moment'][m].mean():11.3f}")

print()
for i in np.argsort(-final_val["ranks"], kind="stable")[:3]:
    like, rows = DATA.val_meta[i], DATA.val_rows[i]
    favourite = rows[np.argmax(final_val["scores"][i])]
    history_likes = int(round(np.expm1(activity[i] * np.log1p(ACTIVITY_CAP))))          # undo the activity formula (capped at 500)
    print(f"user {like.user_id} (about {history_likes} likes so far) really liked: {CATALOGUE.titles[like.row]}")
    print(f"   {final_val['ranks'][i]} of 99 random items scored higher. Model's favourite: {CATALOGUE.titles[favourite]}")
    print("   signals of the liked item: " + ", ".join(f"{n}={v:.2f}" for n, v in zip(FEATURES, DATA.val_x[i, 0])))

## Final exam: `test` (run once)

Do not change `HIDDEN`, `LEARNING_RATE`, `EPOCHS` or anything above after looking at this cell. It scores your trained network and every baseline on `test` and requires your NDCG@10 to be **higher than all of them**. If it is not, go back to Task 2, tune on `val`, and run it again, but never tune on what `test` shows you.

In [ ]:
test_results = {name: exam(fn, "test") for name, fn in BASELINES.items()}
test_results["YOUR NETWORK"] = exam(network_scorer(model), "test")
print(f"{'system':28s} {'NDCG@10':>8s} {'HitRate@10':>11s}")
for name, result in test_results.items():
    print(f"{name:28s} {result['ndcg']:8.4f} {result['hit_rate']:11.3f}")

best_other = max(r["ndcg"] for name, r in test_results.items() if name != "YOUR NETWORK")
assert test_results["YOUR NETWORK"]["ndcg"] > best_other, \
    "on test your network must beat every baseline: go back to Task 2, tune on val (not test), and run again"
print("\nYour network beats every baseline on test.")

## Save the model for InstaKill

A trained model is just numbers. We save them in one `.npz` file together with everything InstaKill needs to use them **exactly like the notebook did**: the standardisation numbers `mu` and `sd`, the order of the signals, and the constants behind the signal formulas. If InstaKill computes `freshness` with a different half-life than the notebook, the network receives inputs it never saw in training and its scores become meaningless. This is **train/serve skew**, one of the most common ways a good model fails in production.

`predict` is a **pure function**: raw signals in, probabilities out, no I/O, no global state. It only needs NumPy, so you can copy it to InstaKill as is. The cell re-loads the file and proves `predict` gives the same numbers as the network you trained.

In [ ]:
MODEL_PATH = Path("like_model.npz")
np.savez(MODEL_PATH, **model, mu=MU, sd=SD, features=np.array(FEATURES),
         half_life_days=HALF_LIFE_DAYS, popularity_days=POPULARITY_DAYS, activity_cap=ACTIVITY_CAP, age_cap_days=AGE_CAP_DAYS)


def predict(saved, features):
    """Raw signals (n, 6) in the order saved['features'] -> P(like) for each row. Pure: no I/O, no globals."""
    x = (features - saved["mu"]) / saved["sd"]
    hidden = np.tanh(x @ saved["W1"] + saved["b1"])
    return sigmoid((hidden @ saved["W2"] + saved["b2"])[:, 0])


saved = dict(np.load(MODEL_PATH))
assert list(saved["features"]) == FEATURES
sample = DATA.val_x.reshape(-1, len(FEATURES))[:2_000]
assert np.allclose(predict(saved, sample), sigmoid(forward(model, standardize(sample))[0])), "the saved model must reproduce the trained one"
print(f"saved {MODEL_PATH}: {count_parameters(model)} trained numbers, {MODEL_PATH.stat().st_size / 1024:.1f} KB on disk")
print("\nthree golden rows for an InstaKill test (signals in the order", FEATURES, "):")
for row in sample[[0, 1, 2]]:
    print("   ", np.round(row, 6).tolist(), "->", round(float(predict(saved, row[None, :])[0]), 6))

## What does one feed request cost?

Serving a feed means: compute the 6 signals for every candidate, run the network on all of them, keep the best `k`. The signals cost `O(C)` (one pass over the `C` candidates); the forward pass is two matrix products, `O(C · 6 · H)`. Both are vectorised, so what you pay in practice is memory traffic, not Python overhead. The cell measures the forward pass on 1 000, 10 000 and 100 000 candidates. Use the numbers in `RESULTS`.

In [ ]:
rng_time = np.random.default_rng(0)
timings = {}
for n in (1_000, 10_000, 100_000):
    random_features = rng_time.random((n, len(FEATURES)))
    timings[n] = mean_ms(lambda: predict(saved, random_features), repeats=20)
    print(f"{n:>8,} candidates: predict takes {timings[n]:8.2f} ms")
print(f"-> 100x more candidates took {timings[100_000] / timings[1_000]:.0f}x longer (linear growth would be about 100x).")
demo_signals_ms = mean_ms(lambda: signals_demo.features(demo_user, all_rows, demo_now), repeats=20)
print(f"computing the 6 signals for all {len(all_rows):,} items of one user: {demo_signals_ms:.1f} ms")

## RESULTS

Replace every placeholder before submission. What goes where:

- `baseline:` NDCG@10 of chronological, popularity, the Week 4 default weights, **your Week 4 weights** and the linear model (val and test).
- `solution:` your `HIDDEN`, `LEARNING_RATE`, `EPOCHS`, and the NDCG@10 and HitRate@10 of your network (val and test). One sentence: did the hidden layer beat the straight line, and by how much?
- `runtime:` training time, the `predict` timings for 1k / 10k / 100k candidates, and the Big-O of one forward pass in your own words.
- `memory / model size:` number of trained parameters and the size of `like_model.npz`.
- `3 failure cases:` your three sentences from the failure analysis (one may be about the group of users that is served worst).
- `what I would try next:` one idea, and which InstaKill problem it would solve.

```text
RESULTS
baseline:
solution:
runtime:
memory / model size:
3 failure cases:
what I would try next:
```

**Next, in InstaKill** (see the assignment overview): copy `like_model.npz` into your repository, copy `predict`, and add `strategy=learned` to `GET /feed`. The notebook's signal formulas are your specification; the golden rows above are your first test.